In [1]:
import pandas as pd
import numpy as np
from PIL import Image
import os
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.tv_tensors import BoundingBoxes
from torchvision.ops import box_iou
import torchvision.transforms.v2 as v2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
import torchmetrics as tm

In [2]:
seed = 42
torch.random.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
root_path = "/home/stefan/ioai-prep/kits/spooky/haunt-me"

batch_size = 64

# Data

In [3]:
class SpookyDataset(Dataset):
    def __init__(self, is_train: bool):
        self.is_train = is_train
        self.dir = f"{root_path}/{'train' if is_train else 'inference'}"
        self.files = os.listdir(self.dir)

        aug_transforms = []
        if is_train:
            aug_transforms = [
                v2.RandomHorizontalFlip(p=0.5),
                v2.RandomVerticalFlip(p=0.3),
                v2.ColorJitter(brightness=0.2, contrast=0.2),
            ]

        final_transforms = [
            v2.Resize((224, 224), antialias=True),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]

        self.transforms = v2.Compose(aug_transforms + final_transforms)

        if is_train:
            self.df = pd.read_csv(f"{root_path}/train_data.csv")

    def __getitem__(self, idx):
        file = f"{self.dir}/{self.files[idx]}"
        image_id = self.files[idx][:-4]

        img = Image.open(file).convert("RGB")

        if not self.is_train:
            img = self.transforms(img)
            return image_id, img

        row = self.df[self.df["image_id"] == image_id].iloc[0]
        bbox_vals = row.iloc[2:6].tolist()
        label = torch.tensor(row.iloc[1], dtype=torch.long)

        bboxes = BoundingBoxes([bbox_vals], format="xyxy", canvas_size=(512, 512))
        img, bboxes = self.transforms(img, bboxes)
        bbox = bboxes[0] if len(bboxes) > 0 else torch.zeros(4, dtype=torch.float32)

        return {"image": img, "label": label, "bbox": bbox}

    def __len__(self):
        return len(self.files)

In [4]:
train_ds = SpookyDataset(is_train=True)
test_ds = SpookyDataset(is_train=False)

n = len(train_ds)
n_train = int(0.8 * n)
n_val = n - n_train

train_ds, val_ds = random_split(
    train_ds, [n_train, n_val], generator=torch.Generator().manual_seed(seed)
)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [5]:
x = next(iter(train_loader))

{k:x[k].shape for k in x.keys()}

{'image': torch.Size([64, 3, 224, 224]),
 'label': torch.Size([64]),
 'bbox': torch.Size([64, 4])}

# Model

In [6]:
def create_model(num_classes=2):
    model = fasterrcnn_mobilenet_v3_large_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = create_model(num_classes=2).to(device)

# Sanity check
model.eval()
with torch.no_grad():
    out = model(x["image"].to(device))
model.train()
print([o["boxes"].shape for o in out])

[torch.Size([46, 4]), torch.Size([20, 4]), torch.Size([25, 4]), torch.Size([89, 4]), torch.Size([21, 4]), torch.Size([0, 4]), torch.Size([12, 4]), torch.Size([100, 4]), torch.Size([72, 4]), torch.Size([3, 4]), torch.Size([29, 4]), torch.Size([44, 4]), torch.Size([15, 4]), torch.Size([15, 4]), torch.Size([35, 4]), torch.Size([19, 4]), torch.Size([100, 4]), torch.Size([26, 4]), torch.Size([1, 4]), torch.Size([6, 4]), torch.Size([75, 4]), torch.Size([27, 4]), torch.Size([25, 4]), torch.Size([8, 4]), torch.Size([14, 4]), torch.Size([100, 4]), torch.Size([3, 4]), torch.Size([6, 4]), torch.Size([0, 4]), torch.Size([13, 4]), torch.Size([31, 4]), torch.Size([31, 4]), torch.Size([22, 4]), torch.Size([100, 4]), torch.Size([22, 4]), torch.Size([1, 4]), torch.Size([10, 4]), torch.Size([14, 4]), torch.Size([20, 4]), torch.Size([36, 4]), torch.Size([13, 4]), torch.Size([3, 4]), torch.Size([43, 4]), torch.Size([14, 4]), torch.Size([100, 4]), torch.Size([20, 4]), torch.Size([4, 4]), torch.Size([64, 4]

# Training

In [7]:
def prepare_targets(labels, bboxes, device):
    targets = []
    for i in range(len(labels)):
        if labels[i] == 1:
            targets.append({
               "boxes": bboxes[i].unsqueeze(0).to(device),
                "labels": torch.tensor([1], dtype=torch.long, device=device),
            })
        else: # No ghost
            targets.append({
                "boxes": torch.zeros((0, 4), dtype=torch.float32, device=device),
                "labels": torch.zeros((0,), dtype=torch.long, device=device),
            })
    return targets

In [8]:
def evaluate(model, loader, score_thresh=0.5):
    all_preds = []
    all_labels = []
    iou_list = []

    model.eval()
    with torch.no_grad():
        for b in loader:
            imgs = b["image"].to(device)
            labels = b["label"].to(device)
            bboxes_gt = b["bbox"].to(device)

            predictions = model(imgs)

            for i, pred in enumerate(predictions):
                scores = pred["scores"]
                high_conf_indices = scores > score_thresh
                has_ghost_pred = high_conf_indices.any().item()
                all_preds.append(int(has_ghost_pred))
                all_labels.append(labels[i].item())

                if labels[i] == 1 and has_ghost_pred:
                    best_box_idx = scores.argmax()
                    box_pred = pred["boxes"][best_box_idx].unsqueeze(0)
                    box_gt = bboxes_gt[i].unsqueeze(0)
                    iou = box_iou(box_pred, box_gt)[0, 0]
                    iou_list.append(iou.item())

    acc = tm.Accuracy(task="binary").to(device)
    all_preds_t = torch.tensor(all_preds, device=device)
    all_labels_t = torch.tensor(all_labels, device=device)

    return {
        "acc": acc(all_preds_t, all_labels_t).item(),
        "iou": np.mean(iou_list).item() if iou_list else 0.0,
    }

In [9]:
lr = 2e-3
epochs = 15

In [10]:
optim = torch.optim.AdamW(model.parameters(), lr=lr)
scaler = torch.amp.GradScaler(device)

In [11]:
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader):
        imgs = batch["image"].to(device)
        labels = batch["label"].long().to(device)
        bboxes = batch["bbox"].to(device)

        targets = prepare_targets(labels, bboxes, device)

        with torch.amp.autocast(device):
            loss_dict = model(imgs, targets)
            loss = sum(loss_dict.values())

        optim.zero_grad()
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optim)
        scaler.update()

        total_loss += loss.item()

    total_loss /= len(train_loader)
    metrics = evaluate(model, val_loader)
    print(f"Epoch {epoch}, loss={total_loss:.3f}, metrics={metrics}")

100%|██████████| 10/10 [00:19<00:00,  1.90s/it]


Epoch 1, loss=0.769, metrics={'acc': 0.543749988079071, 'iou': 0.0}


100%|██████████| 10/10 [00:11<00:00,  1.11s/it]


Epoch 2, loss=0.587, metrics={'acc': 0.8500000238418579, 'iou': 0.7902297744044848}


100%|██████████| 10/10 [00:10<00:00,  1.08s/it]


...

Epoch 13, loss=0.415, metrics={'acc': 0.96875, 'iou': 0.9022650530156331}


100%|██████████| 10/10 [00:10<00:00,  1.05s/it]


Epoch 14, loss=0.368, metrics={'acc': 0.9624999761581421, 'iou': 0.9179972004808484}


100%|██████████| 10/10 [00:10<00:00,  1.05s/it]


Epoch 15, loss=0.402, metrics={'acc': 0.9750000238418579, 'iou': 0.9036421288457535}


# Submission

In [13]:
submission_rows = []
score_thresh = 0.5

model.eval()
with torch.no_grad():
    for image_ids, imgs in tqdm(test_loader, desc="Generating submission"):
        predictions = model(imgs.to(device))

        for img_id, pred in zip(image_ids, predictions):
            scores = pred["scores"]
            boxes = pred["boxes"]

            high_conf_indices = scores > score_thresh
            ghost = 1 if high_conf_indices.any() else 0
            submission_rows.append(
                {"subtaskID": 1, "datapointID": img_id, "answer": str(ghost)}
            )

            if ghost == 1:
                best_box_idx = scores.argmax()
                box = boxes[best_box_idx]
                box_scaled = (box * (512.0 / 224.0)).clamp(0, 512)
                box_str = f"[{int(box_scaled[0])},{int(box_scaled[1])},{int(box_scaled[2])},{int(box_scaled[3])}]"
            else:
                box_str = "[-1,-1,-1,-1]"

            submission_rows.append(
                {"subtaskID": 2, "datapointID": img_id, "answer": box_str}
            )

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv("submission.csv", index=False)
print(f"Submission saved with {len(submission_df)} rows")

Generating submission: 100%|██████████| 4/4 [00:01<00:00,  2.34it/s]

Submission saved with 400 rows
